# Linear Regression for Adsorption-Capacity Prediction

### Materials Informatics Minor Project

This notebook develops baseline **linear regression models** to predict the adsorption capacity, \(q_e\) (mg/g), of activated-carbon/adsorbent materials for water pollutants such as **Pb(II), Cd(II), Methylene Blue, and Congo Red**.

The workflow is designed for a literature-compiled materials dataset (~185 observations). It treats experimental variables such as pollutant concentration, pH, contact time, temperature, adsorbent dosage, and adsorbent/pollutant categories as predictors. The aim is not only prediction, but also interpretation of how experimental and material descriptors are associated with measured adsorption capacity.

> **Important:** The code intentionally detects columns by data type and common name patterns where possible. Once the actual CSV headers are available, review the `TARGET_COLUMN` and any manually specified categorical/numerical columns in the configuration cell.

## 1. Setup — Import Required Libraries

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

sns.set_theme(style='whitegrid', context='notebook')
RANDOM_STATE = 42

print('Libraries imported successfully.')

Libraries imported successfully.


## 2. Configuration

Change `TARGET_COLUMN` if your CSV uses a different name for adsorption capacity. If automatic column detection does not identify your categorical/numerical columns correctly, replace the `None` values with explicit column lists.

In [ ]:
DATA_FILE = 'dataset.csv'

# Common possibilities: 'qe', 'q_e', 'qe_mg_g', 'adsorption_capacity'
TARGET_COLUMN = 'qe'

# Optional manual overrides. Leave as None for automatic detection.
NUMERICAL_COLUMNS = None
CATEGORICAL_COLUMNS = None

# Set to False if your dataset contains columns that should be excluded from modelling.
# Example: EXCLUDE_COLUMNS = ['paper_id', 'author', 'reference']
EXCLUDE_COLUMNS = []

TEST_SIZE = 0.20
CV_FOLDS = 5

print(f'Dataset file: {DATA_FILE}')
print(f'Target column: {TARGET_COLUMN}')

## 3. Load the Literature Dataset

The expected input is a CSV file named `dataset.csv` in the same directory as this notebook. Each row should represent one experimental observation, and the target should be the experimentally reported adsorption capacity \(q_e\) in mg/g.

In [ ]:
df = pd.read_csv(DATA_FILE)

print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
display(df.head())

print('\nColumn names:')
print(df.columns.tolist())

## 4. Initial Data Inspection

Before fitting a model, inspect the units, ranges, and meaning of each variable. For adsorption datasets compiled from multiple papers, differences in experimental conditions and reporting conventions can strongly affect the measured \(q_e\).

In [ ]:
print('Data types:')
display(df.dtypes.to_frame('dtype'))

print('\nSummary statistics for numerical variables:')
display(df.describe(include='all').T)

## 5. Missing-Value Analysis

In [ ]:
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': 100 * df.isna().mean()
}).sort_values('missing_count', ascending=False)

display(missing[missing['missing_count'] > 0])

if missing['missing_count'].sum() == 0:
    print('No missing values detected.')

## 6. Target-Variable Inspection

The target is adsorption capacity, \(q_e\), expressed in mg/g. A histogram and boxplot help identify skewness and possible extreme observations. Extreme values should not be removed automatically: in a literature dataset they may represent genuinely high-performing adsorbents, different test conditions, or inconsistent reporting, so they should be investigated scientifically.

In [ ]:
if TARGET_COLUMN not in df.columns:
    raise KeyError(
        f"Target column '{TARGET_COLUMN}' was not found. "
        f"Available columns are: {df.columns.tolist()}"
    )

target = pd.to_numeric(df[TARGET_COLUMN], errors='coerce')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(target.dropna(), kde=True, ax=axes[0])
axes[0].set_title('Distribution of adsorption capacity ($q_e$)')
axes[0].set_xlabel('$q_e$ (mg/g)')

sns.boxplot(x=target, ax=axes[1])
axes[1].set_title('Boxplot of adsorption capacity')
axes[1].set_xlabel('$q_e$ (mg/g)')
plt.tight_layout()
plt.show()

## 7. Correlation Analysis

Pearson correlation is useful as an initial screening tool for linear association between numerical variables. A strong correlation does **not** establish causation, and correlations can also arise because several experimental variables are coupled.

In [ ]:
numeric_for_corr = df.select_dtypes(include=np.number)

if numeric_for_corr.shape[1] >= 2:
    plt.figure(figsize=(12, 9))
    sns.heatmap(
        numeric_for_corr.corr(),
        cmap='coolwarm',
        center=0,
        annot=True,
        fmt='.2f',
        linewidths=0.5
    )
    plt.title('Pearson Correlation Heatmap')
    plt.tight_layout()
    plt.show()
else:
    print('Not enough numerical columns for a correlation heatmap.')

## 8. Pairplots for Key Numerical Variables

Pairplots can become crowded when many variables are present. The code below selects a small set of common adsorption-process variables when available and always attempts to include the target.

In [ ]:
candidate_keywords = [
    'concentration', 'initial_concentration', 'pH', 'ph',
    'contact_time', 'time', 'temperature', 'temp',
    'dosage', 'dose', 'surface_area', 'pore_volume'
]

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
key_features = []

for col in numeric_cols:
    if col == TARGET_COLUMN:
        continue
    if any(k.lower() in col.lower() for k in candidate_keywords):
        key_features.append(col)

key_features = key_features[:5]

if TARGET_COLUMN in df.columns:
    pairplot_cols = key_features + [TARGET_COLUMN]
    if len(pairplot_cols) >= 2:
        sns.pairplot(df[pairplot_cols].dropna())
        plt.suptitle('Pairplot of Selected Adsorption Variables', y=1.02)
        plt.show()
    else:
        print('Not enough suitable numerical variables for a pairplot.')

print('Selected pairplot features:', key_features)

## 9. Prepare Features and Target

For modelling, the target is separated from the predictor matrix. Obvious identifier/reference columns can be excluded because they do not represent physical or experimental information. Categorical variables such as pollutant type and adsorbent type are converted to numerical indicator variables using one-hot encoding.

In [ ]:
model_df = df.copy()

# Remove user-specified non-predictive columns.
model_df = model_df.drop(columns=[c for c in EXCLUDE_COLUMNS if c in model_df.columns])

# Convert target to numeric and remove rows where the target itself is unavailable.
model_df[TARGET_COLUMN] = pd.to_numeric(model_df[TARGET_COLUMN], errors='coerce')
model_df = model_df.dropna(subset=[TARGET_COLUMN]).copy()

X = model_df.drop(columns=[TARGET_COLUMN])
y = model_df[TARGET_COLUMN]

print('Rows available for modelling:', len(model_df))
print('Predictor columns:', X.columns.tolist())

## 10. Identify Numerical and Categorical Predictors

Automatic detection is used by default. If the actual dataset contains numeric values stored as text, or if a column should be treated as categorical despite being numeric-coded, set the manual lists in the configuration cell.

In [ ]:
if NUMERICAL_COLUMNS is None:
    numerical_features = X.select_dtypes(include=np.number).columns.tolist()
else:
    numerical_features = [c for c in NUMERICAL_COLUMNS if c in X.columns]

if CATEGORICAL_COLUMNS is None:
    categorical_features = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
else:
    categorical_features = [c for c in CATEGORICAL_COLUMNS if c in X.columns]

# Prevent accidental overlap.
categorical_features = [c for c in categorical_features if c not in numerical_features]

print('Numerical features:')
print(numerical_features)
print('\nCategorical features:')
print(categorical_features)

## 11. Train-Test Split

An 80/20 split is used. `random_state=42` makes the split reproducible. The test set is kept separate from model fitting and is used only for the final hold-out evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print(f'Training observations: {len(X_train)}')
print(f'Test observations:     {len(X_test)}')

## 12. Preprocessing Pipeline

Numerical variables are median-imputed and standardized. Categorical variables are imputed with the most frequent category and then one-hot encoded. Keeping preprocessing inside a scikit-learn `Pipeline` prevents information from the test set from leaking into the training process.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

print('Preprocessing pipeline created.')

## 13. Define Linear, Ridge, and Lasso Models

**Linear Regression** provides the baseline least-squares model. **Ridge** adds L2 regularization, while **Lasso** adds L1 regularization. Regularization can be useful when many correlated descriptors or one-hot encoded categories make the coefficient estimates unstable.

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.01, max_iter=10000)
}

pipelines = {
    name: Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    for name, model in models.items()
}


## 14. Fit Models and Evaluate on Train/Test Data

The three metrics used are:

- **R²:** proportion of variance in measured \(q_e\) explained by the model.
- **RMSE:** square-root mean squared error, in the same units as \(q_e\) (mg/g); it penalizes larger errors more strongly.
- **MAE:** mean absolute error, also in mg/g; it is easier to interpret as a typical absolute prediction error.


In [ ]:
results = []
predictions = {}
fitted_pipelines = {}

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

    y_train_pred = pipe.predict(X_train)
    y_test_pred = pipe.predict(X_test)
    predictions[name] = {'train': y_train_pred, 'test': y_test_pred}

    results.append({
        'Model': name,
        'Train R²': r2_score(y_train, y_train_pred),
        'Test R²': r2_score(y_test, y_test_pred),
        'Train RMSE (mg/g)': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'Test RMSE (mg/g)': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'Train MAE (mg/g)': mean_absolute_error(y_train, y_train_pred),
        'Test MAE (mg/g)': mean_absolute_error(y_test, y_test_pred)
    })

results_df = pd.DataFrame(results).set_index('Model')
display(results_df.round(4))

## 15. Actual vs Predicted Adsorption Capacity

Points close to the 1:1 line indicate agreement between experimentally measured and model-predicted adsorption capacities. Systematic deviations can reveal under-prediction at high capacity, heteroscedasticity, or nonlinear behaviour.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, pred) in zip(axes, predictions.items()):
    ax.scatter(y_test, pred['test'], alpha=0.75)
    min_val = min(y_test.min(), pred['test'].min())
    max_val = max(y_test.max(), pred['test'].max())
    ax.plot([min_val, max_val], [min_val, max_val], linestyle='--')
    ax.set_xlabel('Measured $q_e$ (mg/g)')
    ax.set_ylabel('Predicted $q_e$ (mg/g)')
    ax.set_title(name)

plt.suptitle('Actual vs Predicted Adsorption Capacity — Test Set', y=1.02)
plt.tight_layout()
plt.show()

## 16. Residual Analysis

Residuals are defined as measured \(q_e\) minus predicted \(q_e\). For a well-specified linear model, residuals should ideally be scattered around zero without a strong systematic pattern. Curvature or changing residual spread suggests that a purely linear relationship may be insufficient.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, pred) in zip(axes, predictions.items()):
    residuals = y_test - pred['test']
    ax.scatter(pred['test'], residuals, alpha=0.75)
    ax.axhline(0, linestyle='--')
    ax.set_xlabel('Predicted $q_e$ (mg/g)')
    ax.set_ylabel('Residual (mg/g)')
    ax.set_title(name)

plt.suptitle('Residual Plots — Test Set', y=1.02)
plt.tight_layout()
plt.show()

## 17. Regression Coefficients / Feature Importance

For linear models, coefficients describe the direction and magnitude of the modelled association while all numerical features are standardized. Therefore, coefficients for standardized numerical variables are directly comparable in scale. One-hot encoded categorical coefficients are interpreted relative to the omitted/reference category.

> **Interpretation caution:** A large coefficient does not prove a mechanistic adsorption effect. Literature datasets can contain confounding between pollutant type, adsorbent chemistry, concentration, pH, and experimental protocol.

In [ ]:
def get_feature_names_and_coefficients(pipe):
    fitted_preprocessor = pipe.named_steps['preprocessor']
    model = pipe.named_steps['model']
    feature_names = fitted_preprocessor.get_feature_names_out()
    coefficients = np.ravel(model.coef_)
    coef_df = pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': coefficients,
        'Absolute_Coefficient': np.abs(coefficients)
    }).sort_values('Absolute_Coefficient', ascending=False)
    return coef_df

coef_tables = {}

for name in fitted_pipelines:
    coef_tables[name] = get_feature_names_and_coefficients(fitted_pipelines[name])
    print(f'\n{name} — top coefficients')
    display(coef_tables[name].head(20).round(4))

In [ ]:
# Plot the 15 largest-magnitude coefficients for Linear Regression.
coef_plot_df = coef_tables['Linear Regression'].head(15).sort_values('Coefficient')

plt.figure(figsize=(10, 7))
plt.barh(coef_plot_df['Feature'], coef_plot_df['Coefficient'])
plt.axvline(0, linestyle='--')
plt.xlabel('Regression coefficient')
plt.ylabel('Encoded feature')
plt.title('Largest-Magnitude Linear Regression Coefficients')
plt.tight_layout()
plt.show()

## 18. Five-Fold Cross-Validation

The hold-out test set gives one estimate of generalization performance. Five-fold cross-validation provides additional evidence by repeatedly training on four folds and evaluating on the remaining fold. Here, the reported values are cross-validated R² scores.

In [ ]:
cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

cv_results = []

for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X, y, cv=cv, scoring='r2')
    cv_results.append({
        'Model': name,
        'CV R² Mean': scores.mean(),
        'CV R² Std': scores.std(),
        'Fold 1': scores[0],
        'Fold 2': scores[1],
        'Fold 3': scores[2],
        'Fold 4': scores[3],
        'Fold 5': scores[4]
    })

cv_df = pd.DataFrame(cv_results).set_index('Model')
display(cv_df.round(4))

## 19. Compact Model Comparison

This table combines the main hold-out and cross-validation results. Use the metrics to understand predictive behaviour rather than treating a single metric as a complete description of model quality.

In [ ]:
comparison = results_df.join(cv_df[['CV R² Mean', 'CV R² Std']])
display(comparison.round(4))

## 20. Materials-Engineering Interpretation Checklist

When discussing these results in the minor-project report, consider:

1. **Pollutant chemistry:** Pb(II), Cd(II), Methylene Blue, and Congo Red differ substantially in charge, molecular size, hydration, and interaction mechanisms with carbon surfaces.
2. **Adsorbent properties:** Surface area, pore-size distribution, surface functional groups, activation conditions, and precursor material can influence adsorption capacity.
3. **Solution chemistry:** pH can change both adsorbent surface charge and pollutant speciation/ionization, so its coefficient should be interpreted in chemical context.
4. **Initial concentration and dosage:** These variables are experimentally coupled to the definition and measured value of \(q_e\), so their relationships with the target require careful interpretation.
5. **Literature heterogeneity:** Different papers may use different equilibrium times, analytical methods, particle sizes, activation routes, temperature conditions, and reporting conventions.
6. **Linear-model limitation:** Adsorption isotherms and kinetic behaviour are frequently nonlinear. A linear model is therefore a useful interpretable baseline, not necessarily the final predictive model.
7. **Data leakage:** Do not use experimentally derived quantities that directly contain the target (or were calculated using \(q_e\)) as predictors.
8. **Sample size:** With roughly 185 observations, keep the number of independent descriptors under control and use cross-validation to assess robustness.

## 21. Optional: Export Predictions for Further Analysis

This creates a table containing the measured test-set capacity and predictions from all three models. It can be useful for later error analysis or for comparing linear baselines against nonlinear models such as Random Forest, Gradient Boosting, XGBoost, or neural networks.

In [ ]:
prediction_export = X_test.copy()
prediction_export['Measured_qe_mg_g'] = y_test.values

for name, pred in predictions.items():
    safe_name = name.lower().replace(' ', '_')
    prediction_export[f'Predicted_qe_{safe_name}'] = pred['test']

prediction_export.to_csv('linear_model_test_predictions.csv', index=False)
print("Saved: linear_model_test_predictions.csv")
display(prediction_export.head())

# End of Notebook

Once the actual dataset headers are available, update the configuration cell—especially `TARGET_COLUMN`, and manually specify numerical/categorical columns if automatic detection is inappropriate. Then run the notebook from top to bottom.